# Euler Diagrams with Rectangles — MILP

**Variables**
- Element positions `(ex_i, ey_i)` — movable small squares
- Set rectangles `(rx_j, ry_j, rw_j, rh_j)` — bottom-left corner + width/height

**Constraints** (from a boolean membership matrix)
- *Containment*: element `i` ∈ set `j` → 4 linear inequalities, no binaries
- *Exclusion*: element `i` ∉ set `j` → disjunctive (OR of 4 separating planes), 4 binaries per pair

**Objective**: minimize total perimeter `∑ 2(rw_j + rh_j)`

In [ ]:
import numpy as np
import pulp
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import to_rgba

In [ ]:
def solve_euler_rectangles(membership, r=0.15, offset=0.2, layout_size=5.0):
    """Find element positions and set rectangles forming an Euler diagram.

    Args:
        membership: (N, S) bool array — membership[i, j] = True if element i ∈ set j
        r: half-side of each element square
        offset: minimum gap enforced at every border (containment: element ≥ offset
            from each wall; exclusion/non-overlap: borders ≥ offset apart)
        layout_size: bounding box for all variables; big-M is derived from this

    Returns:
        dict with keys:
          status: PuLP status string
          element_positions: (N, 2) array of (x, y) centres
          rectangles: (S, 4) array of [x, y, w, h] (bottom-left + size)
    """
    N, S = membership.shape
    L = layout_size
    M = 2 * L + 2 * r + offset

    prob = pulp.LpProblem("euler_rectangles", pulp.LpMinimize)

    ex = [pulp.LpVariable(f"ex_{i}", lowBound=0, upBound=L) for i in range(N)]
    ey = [pulp.LpVariable(f"ey_{i}", lowBound=0, upBound=L) for i in range(N)]

    rx = [pulp.LpVariable(f"rx_{j}", lowBound=0, upBound=L) for j in range(S)]
    ry = [pulp.LpVariable(f"ry_{j}", lowBound=0, upBound=L) for j in range(S)]
    rw = [pulp.LpVariable(f"rw_{j}", lowBound=2 * (r + offset), upBound=L) for j in range(S)]
    rh = [pulp.LpVariable(f"rh_{j}", lowBound=2 * (r + offset), upBound=L) for j in range(S)]

    prob += pulp.lpSum(2 * (rw[j] + rh[j]) for j in range(S))

    # Containment / exclusion: element <-> set rectangle
    for i in range(N):
        for j in range(S):
            if membership[i, j]:
                # element square must be >= offset from each wall
                prob += ex[i] - r >= rx[j]           + offset
                prob += ex[i] + r <= rx[j] + rw[j]   - offset
                prob += ey[i] - r >= ry[j]           + offset
                prob += ey[i] + r <= ry[j] + rh[j]   - offset
            else:
                # element square must be >= offset outside the rectangle
                bL = pulp.LpVariable(f"bL_{i}_{j}", cat="Binary")
                bR = pulp.LpVariable(f"bR_{i}_{j}", cat="Binary")
                bB = pulp.LpVariable(f"bB_{i}_{j}", cat="Binary")
                bT = pulp.LpVariable(f"bT_{i}_{j}", cat="Binary")
                prob += bL + bR + bB + bT >= 1
                prob += ex[i] + r <= rx[j]          - offset + M * (1 - bL)
                prob += ex[i] - r >= rx[j] + rw[j]  + offset - M * (1 - bR)
                prob += ey[i] + r <= ry[j]          - offset + M * (1 - bB)
                prob += ey[i] - r >= ry[j] + rh[j]  + offset - M * (1 - bT)

    # Element non-overlap: every pair of squares must be >= offset apart
    for i in range(N):
        for k in range(i + 1, N):
            bL = pulp.LpVariable(f"ebL_{i}_{k}", cat="Binary")
            bR = pulp.LpVariable(f"ebR_{i}_{k}", cat="Binary")
            bB = pulp.LpVariable(f"ebB_{i}_{k}", cat="Binary")
            bT = pulp.LpVariable(f"ebT_{i}_{k}", cat="Binary")
            prob += bL + bR + bB + bT >= 1
            prob += ex[i] + 2 * r + offset <= ex[k] + M * (1 - bL)
            prob += ex[k] + 2 * r + offset <= ex[i] + M * (1 - bR)
            prob += ey[i] + 2 * r + offset <= ey[k] + M * (1 - bB)
            prob += ey[k] + 2 * r + offset <= ey[i] + M * (1 - bT)

    solver = pulp.HiGHS(msg=False)
    prob.solve(solver)

    elem_pos = np.array([[pulp.value(ex[i]), pulp.value(ey[i])] for i in range(N)])
    rects = np.array([[pulp.value(rx[j]), pulp.value(ry[j]), pulp.value(rw[j]), pulp.value(rh[j])] for j in range(S)])

    return {"status": pulp.LpStatus[prob.status], "element_positions": elem_pos, "rectangles": rects}

In [ ]:
def plot_euler(membership, result, r=0.15, set_labels=None, colors=None):
    N, S = membership.shape
    if set_labels is None:
        set_labels = [f"Set {j}" for j in range(S)]
    if colors is None:
        colors = plt.cm.tab10(np.linspace(0, 0.9, S))

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect("equal")

    # draw rectangles back-to-front (largest first) so smaller ones are visible
    areas = result["rectangles"][:, 2] * result["rectangles"][:, 3]
    for j in np.argsort(-areas):
        x, y, w, h = result["rectangles"][j]
        ax.add_patch(patches.Rectangle(
            (x, y), w, h, linewidth=2,
            edgecolor=colors[j], facecolor=to_rgba(colors[j], 0.12), zorder=2
        ))
        ax.text(x + w / 2, y + h + 0.05, set_labels[j],
                ha="center", va="bottom", color=colors[j], fontsize=11, fontweight="bold")

    for i in range(N):
        ex, ey = result["element_positions"][i]
        ax.add_patch(patches.Rectangle(
            (ex - r, ey - r), 2 * r, 2 * r,
            linewidth=1, edgecolor="black", facecolor="#555", zorder=3
        ))
        ax.text(ex, ey, str(i), ha="center", va="center",
                fontsize=7, color="white", zorder=4)

    # auto-fit: derive bounds from elements and rectangles
    rects = result["rectangles"]
    all_x = np.concatenate([rects[:, 0], rects[:, 0] + rects[:, 2], result["element_positions"][:, 0]])
    all_y = np.concatenate([rects[:, 1], rects[:, 1] + rects[:, 3], result["element_positions"][:, 1]])
    pad = 0.4
    ax.set_xlim(all_x.min() - pad, all_x.max() + pad)
    ax.set_ylim(all_y.min() - pad, all_y.max() + pad)

    ax.set_title(f"status: {result['status']}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

## Example 1 — two disjoint sets

In [ ]:
membership = np.array([
    [1, 0],
    [1, 0],
    [1, 0],
    [0, 1],
    [0, 1],
    [0, 1],
], dtype=bool)

result = solve_euler_rectangles(membership)
plot_euler(membership, result, set_labels=["A", "B"])

## Example 2 — nested sets (A ⊆ B)

In [ ]:
membership = np.array([
    [1, 1],  # ∈ A and B
    [1, 1],
    [0, 1],  # ∈ B only
    [0, 1],
    [0, 1],
], dtype=bool)

result = solve_euler_rectangles(membership)
plot_euler(membership, result, set_labels=["A", "B"])

## Example 3 — A and B disjoint, both contained in C

In [ ]:
membership = np.array([
    [1, 0, 1],  # ∈ A, C
    [1, 0, 1],
    [1, 0, 1],
    [0, 1, 1],  # ∈ B, C
    [0, 1, 1],
    [0, 1, 1],
    [0, 0, 1],  # ∈ C only
], dtype=bool)

result = solve_euler_rectangles(membership)
plot_euler(membership, result, set_labels=["A", "B", "C"])